# Five machine learning models on real market data

A replication of Krauss, Do and Huck (2017) with decision trees, random forests and XGBoost on a
survivorship free panel of S&P 500 stocks, then PCA on the Treasury curve and Lasso on a wide
factor panel.

Runs top to bottom. Prices come from Yahoo Finance, rates and macro from FRED, and index
membership from the Wikipedia revision history. No API key is needed anywhere.

**Expect ten to fifteen minutes on the first run**, almost all of it downloading. Two caches
are written to disk (`membership.json` and `precos.parquet`), so later runs take about three
minutes.

Full write up: https://davidariasfinance.com/scripts/machine-learning-for-finance/

## Step 0. Install and import

Everything the notebook needs, in one place, before anything runs.

In [ ]:
%pip install -q yfinance pandas numpy scikit-learn xgboost matplotlib requests lxml pyarrow

In [ ]:
import io, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf

import matplotlib
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, roc_curve, r2_score
import xgboost as xgb

START, END   = "2007-01-01", "2015-12-31"   # equity panel: the overlap with the paper
TODAY        = "2026-08-19"                 # rates and ETFs run to the present instead
TRAIN, TRADE = 750, 250                     # sliding window, in trading days
PERIODS = list(range(1, 21)) + list(range(40, 241, 20))   # 31 lookbacks
HEADERS = {"User-Agent": "Mozilla/5.0 (research)"}
WIKI_API = "https://en.wikipedia.org/w/api.php"

print("ready")

## Step 1. One data helper

FRED is read through its public CSV endpoint, which needs no key. Prices come from yfinance
with `auto_adjust=True`, so they already account for splits and dividends.

In [ ]:
def fred(series):
    """FRED through the public CSV endpoint, no API key required."""
    out = {}
    for s in series:
        r = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                         params={"id": s}, timeout=90)
        r.raise_for_status()
        d = pd.read_csv(io.StringIO(r.text))
        d.columns = ["date", s]
        d["date"] = pd.to_datetime(d["date"])
        out[s] = pd.to_numeric(d.set_index("date")[s], errors="coerce")
    return pd.DataFrame(out)

## Step 2. A survivorship free universe

Taking today's S&P 500 list and running it back to 2007 is the most common way to produce a
flattering backtest: every company that went bankrupt or was acquired is missing, so the sample
is made only of survivors.

Wikipedia keeps a full revision history of its S&P 500 page, so the membership as it actually
stood on any past date can be read back from the revision that was live then. Snapshots every
six months give the real membership through time, delisted names included.

**Slow cell.** Forty revisions with a polite throttle. Cached to `membership.json` afterwards.

In [ ]:
CACHE = Path("membership.json")

def members_on(date):
    p = {"action": "query", "prop": "revisions", "titles": "List of S&P 500 companies",
         "rvlimit": 1, "rvstart": f"{date}T00:00:00Z", "rvdir": "older",
         "rvprop": "ids|timestamp", "format": "json", "formatversion": 2}
    for attempt in range(4):
        resp = requests.get(WIKI_API, params=p, headers=HEADERS, timeout=90)
        try:
            j = resp.json(); break
        except ValueError:                       # rate limited, back off
            time.sleep(3 * (attempt + 1))
    else:
        return []
    rev = j["query"]["pages"][0].get("revisions")
    if not rev:
        return []
    html = requests.get("https://en.wikipedia.org/w/index.php",
                        params={"oldid": rev[0]["revid"]}, headers=HEADERS, timeout=90).text
    try:
        big = [t for t in pd.read_html(io.StringIO(html)) if t.shape[0] > 300]
    except ValueError:
        return []
    if not big:
        return []
    col = [c for c in big[0].columns if str(c).lower().startswith(("symbol", "ticker"))]
    if not col:
        return []
    tick = (big[0][col[0]].astype("string").fillna("").str.strip().str.upper()
            .str.replace(".", "-", regex=False))          # BRK.B -> BRK-B, Yahoo format
    return sorted({x for x in tick.tolist()               # fillna: some revisions carry NaN
                   if isinstance(x, str) and x.isascii()
                   and 1 <= len(x) <= 6 and x.replace("-", "").isalpha()})


snapshots = json.loads(CACHE.read_text()) if CACHE.exists() else {}
wanted = [f"{y}-{m}-01" for y in range(2007, 2027) for m in ("01", "07")]
for d in [d for d in wanted if d <= END and d not in snapshots]:
    got = members_on(d)
    if got:
        snapshots[d] = got
        CACHE.write_text(json.dumps(snapshots, indent=1))
    time.sleep(1.2)

universe = sorted({t for lst in snapshots.values() for t in lst})
print(f"{len(snapshots)} snapshots, {len(universe)} tickers ever in the index")

In [ ]:
sizes = [len(snapshots[d]) for d in sorted(snapshots)]
seen, union = set(), []
for d in sorted(snapshots):
    seen |= set(snapshots[d])
    union.append(len(seen))
x = pd.to_datetime(sorted(snapshots))

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(x, union, color="#7048e8", lw=2.2, label="tickers seen at least once")
ax.plot(x, sizes, color="#2f5db0", lw=2.0, label="members in that snapshot")
ax.fill_between(x, sizes, union, color="#7048e8", alpha=0.08)
ax.set_ylim(0, max(union) * 1.12)
ax.set_title("Every snapshot holds about 500 names, and the union keeps growing", loc="left")
ax.set_ylabel("tickers")
ax.legend(frameon=False, loc="lower left")
plt.show()

print(f"{union[-1]} ever members, {sizes[-1]} today, "
      f"{union[-1] - sizes[-1]} that left and would vanish from a naive sample")

## Step 3. Prices, and the point in time mask

Every ticker that was ever a member gets downloaded, including the ones that no longer trade.
Coverage is reported honestly: Yahoo no longer serves history for many long delisted names, so
the correction is a large improvement rather than a complete one.

Each stock then contributes rows only on the dates it was genuinely a member.

**Slow cell.** Cached to `precos.parquet` afterwards.

In [ ]:
PRICES = Path("precos.parquet")

if PRICES.exists():
    px = pd.read_parquet(PRICES).astype("float32")
else:
    parts = []
    for i in range(0, len(universe), 120):
        d = yf.download(universe[i:i + 120], start=START, end=END,
                        auto_adjust=True, progress=False)
        if isinstance(d.columns, pd.MultiIndex):
            d = d["Close"]
        parts.append(d)
    px = pd.concat(parts, axis=1).sort_index()
    px = px.loc[:, ~px.columns.duplicated()].dropna(axis=1, how="all")
    px.to_parquet(PRICES)

# a cache written by a wider run would silently change the number of windows
px = px.loc[(px.index >= START) & (px.index <= END)]

marks = sorted(snapshots)
mask = pd.DataFrame(False, index=px.index, columns=px.columns)
for i, d0 in enumerate(marks):
    d1 = marks[i + 1] if i + 1 < len(marks) else "2100-01-01"
    mask.loc[(px.index >= d0) & (px.index < d1),
             [t for t in snapshots[d0] if t in mask.columns]] = True

print(f"{px.shape[1]} of {len(universe)} tickers with data "
      f"({px.shape[1]/len(universe):.0%} coverage)")
print(f"median members on a day: {int(mask.sum(axis=1).median())}")

In [ ]:
on_paper = pd.Series(index=px.index, dtype=float)
for i, d0 in enumerate(marks):
    d1 = marks[i + 1] if i + 1 < len(marks) else "2100-01-01"
    on_paper[(px.index >= d0) & (px.index < d1)] = len(snapshots[d0])

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.plot(px.index, on_paper, color="#9aa1ae", lw=1.6, label="index members that day")
ax.plot(px.index, mask.sum(axis=1), color="#0ca678", lw=1.8,
        label="of those, with usable Yahoo history")
ax.fill_between(px.index, mask.sum(axis=1), on_paper, color="#d6336c", alpha=0.10)
ax.set_xlim(pd.Timestamp(marks[0]), px.index[-1])
ax.set_ylim(0, 560)
ax.set_title("Point in time membership, day by day", loc="left")
ax.set_ylabel("stocks in the panel")
ax.legend(frameon=False, loc="lower right")
plt.show()

## Step 4. Features, and a cross sectional target

Asking "will the market go up" hands a model an upward drift it can free ride on. Krauss asks
something relative instead: **will this stock beat the median of its peers tomorrow**. Half the
universe beats the median every day by construction, so the base rate sits at 0.50 and no dumb
rule is worth beating.

Features are 31 cumulative returns and nothing else: every lookback from 1 to 20 trading days,
then 40, 60 and so on to 240. No RSI, no moving average distance, no volatility. Short lookbacks
carry reversal, long ones carry momentum, and the models work out which applies where.

Each column is then standardised **across the cross section of its own day**. A 2% return in
October 2008 and a 2% return in a quiet stretch of 2014 mean different things, while "1.3
standard deviations above today's average" means the same thing in both.

In [ ]:
frames = {}
for m in PERIODS:
    r  = (px / px.shift(m) - 1).where(mask)          # cumulative return over m days
    mu = r.mean(axis=1)                              # today's cross sectional mean
    sd = r.std(axis=1).replace(0, np.nan)
    frames[f"r_{m}"] = r.sub(mu, axis=0).div(sd, axis=0).astype("float32")

r1 = (px.shift(-1) / px - 1).where(mask)             # tomorrow's return
y_all = (r1.rank(axis=1, pct=True) > 0.5).astype("float32").where(r1.notna())

def stack(window):
    # long format for one slice of dates: one row per stock per day
    X = pd.concat({n: f.loc[window].stack() for n, f in frames.items()}, axis=1)
    y = y_all.loc[window].stack().reindex(X.index)
    ok = y.notna() & X.notna().all(axis=1)
    return X[ok].astype("float32"), y[ok].astype("int8")

first = px.index[max(PERIODS):max(PERIODS) + TRAIN]
X0, y0 = stack(first)
print(f"{len(X0):,} rows x {X0.shape[1]} features")
print(f"window {first[0].date()} to {first[-1].date()}, target mean {y0.mean():.4f}")

Fitting one shallow tree on that first window, before any of the machinery below, already shows
the whole problem in one picture. Read the leaves: most of them sit between 0.49 and 0.52, which
is why AUC lands near 0.51, and the one useful leaf holds 0.584 on well under one percent of the
rows. A strategy that trades only the extremes is built to hold exactly that leaf.

In [ ]:
from sklearn.tree import plot_tree

shallow = DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, random_state=0)
shallow.fit(X0, y0)

fig, ax = plt.subplots(figsize=(13.5, 5.4))
plot_tree(shallow, feature_names=list(X0.columns), class_names=["below", "above"],
          filled=False, impurity=False, proportion=True, rounded=False, fontsize=8, ax=ax)
ax.set_title("First three levels of the fitted tree, window 1", loc="left")
plt.show()

## Step 5. Replicating a published strategy

Everything from here to Step 7 rebuilds one specific paper: *Deep neural networks,
gradient-boosted trees, random forests: Statistical arbitrage on the S&P 500*, by Krauss, Do and
Huck, European Journal of Operational Research, 2017. Only the three tree based models are
reproduced, since neural networks belong in a separate post.

What they report, before costs, on 1992 to 2015: **0.43% a day** for the random forest at
t = 14.93, and 0.37% for gradient boosting. After 0.05% per half turn in costs the forest keeps
0.23% a day at t = 7.91, a Sharpe of 1.90.

Three choices carry the method:

1. **Sliding window.** Train on 750 days, trade the next 250, step forward 250 and refit. No model
   ever sees a day it was fitted on, and each refit adapts to the regime it is about to trade.
2. **One day horizon.** Predict tomorrow, not next week.
3. **Trade the extremes only.** Buy the 10 highest probabilities of the day, sell the 10 lowest,
   discard the middle. Most of the ranking is noise, and a long short book never touches it.

In [ ]:
MODELS = {
    "tree":     lambda: DecisionTreeClassifier(max_depth=8, min_samples_leaf=500,
                                               random_state=0),
    "forest":   lambda: RandomForestClassifier(n_estimators=60, max_depth=12,
                                               min_samples_leaf=200, max_features="sqrt",
                                               n_jobs=3, random_state=0),
    "boosting": lambda: xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                          subsample=0.8, colsample_bytree=0.8, n_jobs=3,
                                          tree_method="hist", max_bin=96,
                                          eval_metric="logloss"),
}
K = 10                                    # positions per leg
starts = list(range(max(PERIODS), len(px.index) - TRAIN - TRADE, TRADE))
rows, aucs, roc_data, plan = [], [], {}, []

for w, i0 in enumerate(starts, 1):
    tr_days = px.index[i0:i0 + TRAIN]
    op_days = px.index[i0 + TRAIN:i0 + TRAIN + TRADE]
    Xtr, ytr = stack(tr_days)
    Xop, yop = stack(op_days)
    Atr, Aop = Xtr.to_numpy("float32"), Xop.to_numpy("float32")

    for name, make in MODELS.items():
        model = make().fit(Atr, ytr.to_numpy())
        p_op  = pd.Series(model.predict_proba(Aop)[:, 1], index=Xop.index)
        p_tr  = model.predict_proba(Atr)[:, 1]
        aucs.append((w, name, roc_auc_score(ytr, p_tr), roc_auc_score(yop, p_op)))
        if w == len(starts):
            roc_data[name] = (ytr.to_numpy(), p_tr, yop.to_numpy(), p_op.to_numpy())

        # portfolio: long the K best probabilities of each day, short the K worst
        for d, g in p_op.groupby(level=0):
            if len(g) < 2 * K:
                continue
            o = g.sort_values(ascending=False)
            longs  = o.index[:K].get_level_values(1)
            shorts = o.index[-K:].get_level_values(1)
            rows.append((w, name, d,
                         float(r1.loc[d, longs].mean() - r1.loc[d, shorts].mean())))

    plan.append((w, tr_days[0], tr_days[-1], op_days[0], op_days[-1],
                 len(ytr), len(yop)))
    print(f"window {w}/{len(starts)}  trades {op_days[0].date()} to {op_days[-1].date()}")
    del Xtr, ytr, Xop, yop, Atr, Aop

port = pd.DataFrame(rows, columns=["window", "model", "date", "ret"])
print(f"{len(port):,} model days recorded")

Before reading any result, look at what the loop actually scheduled. Blue is fitted on, orange is
traded and scored, and consecutive blue blocks overlap by two years, which is why five windows are
not five independent draws.

In [ ]:
from matplotlib.patches import Patch

sched = pd.DataFrame(plan, columns=["window", "tr0", "tr1", "op0", "op1", "n_tr", "n_op"])

fig, ax = plt.subplots(figsize=(12, 4.0))
for i, r in sched.iterrows():
    y = len(sched) - i
    ax.barh(y, (r["tr1"] - r["tr0"]).days, left=r["tr0"], height=0.52, color="#2f5db0", alpha=0.85)
    ax.barh(y, (r["op1"] - r["op0"]).days, left=r["op0"], height=0.52, color="#d97706")
    ax.text(r["tr0"], y + 0.42, f"window {r['window']}", fontsize=9.5)
ax.set_yticks([])
ax.set_ylim(0.3, len(sched) + 1.1)
ax.set_title("Train on 750 days, trade the next 250, step forward and refit", loc="left")
ax.legend(handles=[Patch(color="#2f5db0", alpha=0.85, label="750 days of training"),
                   Patch(color="#d97706", label="250 days traded, the only part ever scored")],
          frameon=False, loc="lower left")
plt.show()

fig, ax = plt.subplots(figsize=(12, 4.4))
p = np.arange(len(sched))
ax.bar(p - 0.19, sched["n_tr"] / 1000, 0.38, color="#2f5db0", label="training rows")
ax.bar(p + 0.19, sched["n_op"] / 1000, 0.38, color="#d97706", label="trading rows")
ax.set_xticks(p, [f"window {i}" for i in sched["window"]])
ax.set_title("One row is one stock on one day, and every window refits on more of them", loc="left")
ax.set_ylabel("rows, thousands")
ax.legend(frameon=False)
plt.show()

view = sched.copy()
for c in ("tr0", "tr1", "op0", "op1"):
    view[c] = view[c].dt.date
print(view.to_string(index=False))

## Step 6. Results, window by window

Five windows, each trained on 750 days and traded on the 250 that follow. Every number below
comes from the trading columns only, so no model is ever scored on data it was fitted on. A
window by window breakdown matters more than the average, because a strategy carried by one lucky
year looks identical to a real one once you take the mean.

In [ ]:
by_w = port.pivot_table(index="window", columns="model", values="ret", aggfunc="mean") * 100
by_w = by_w[["tree", "forest", "boosting"]].round(3)
span = port.groupby("window")["date"].agg(["min", "max"])
by_w.insert(0, "trades",
            span["min"].dt.date.astype(str) + " to " + span["max"].dt.date.astype(str))
print("return per day of the long short book, k = 10, in percent")
print(by_w.to_string())

print()
for name in MODELS:
    v = port.loc[port.model == name, "ret"].to_numpy()
    t = v.mean() / (v.std(ddof=1) / np.sqrt(len(v)))
    print(f"{name:9} {v.mean()*100:+.3f}%/day   annual {((1 + v.mean())**252 - 1)*100:6.1f}%"
          f"   t = {t:5.2f}   windows positive {int((by_w[name] > 0).sum())}/{len(by_w)}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.0))
for name, colour in zip(MODELS, ("#d97706", "#0ca678", "#7048e8")):
    v = port.loc[port.model == name, "ret"].to_numpy()
    ax.plot(np.cumprod(1 + v), color=colour, lw=1.8, label=f"{name}, k=10")
ax.axhline(1, color="#9aa1ae", lw=1.2)
ax.set_yscale("log")
ax.set_title("Long short portfolio, k = 10, before transaction costs", loc="left")
ax.set_xlabel("trading days out of sample")
ax.set_ylabel("growth of 1")
ax.legend(frameon=False)
plt.show()

### What AUC is, and why it looks so bad here

AUC asks a single question: take one stock that did beat the median and one that did not, and how
often does the model give the winner the higher probability? 0.50 means guessing, 1.00 means never
wrong. Accuracy answers a different question, how many calls land on the right side of a cutoff,
and an unbalanced target flatters it. Here the target is balanced 50/50 by construction, so AUC is
the cleaner measure.

Plotting the training curve against the test curve shows how much each model memorised. A gap is
expected and is not a defect. A test curve sitting on the diagonal while the training curve bulges
is the normal picture for a weak signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
for ax, (name, (y_tr, p_tr, y_te, p_te)) in zip(axes, roc_data.items()):
    for y_, p_, style, tag in [(y_tr, p_tr, "--", "train"), (y_te, p_te, "-", "test")]:
        fpr, tpr, _ = roc_curve(y_, p_)
        ax.plot(fpr, tpr, ls=style, lw=1.8, color="#0ca678",
                label=f"{tag}  {roc_auc_score(y_, p_):.3f}")
    ax.plot([0, 1], [0, 1], ls=":", color="#9aa1ae")
    ax.set_title(name, loc="left")
    ax.set_xlabel("false positive rate")
    ax.legend(frameon=False, loc="lower right")
axes[0].set_ylabel("true positive rate")
plt.suptitle("ROC in sample and out of sample, last window", x=0.09, ha="left")
plt.show()

tab = pd.DataFrame(aucs, columns=["window", "model", "train", "test"])
print(tab.groupby("model")[["train", "test"]].mean().round(4).to_string())

Out of sample the models score about **0.51**, a hair above guessing. Over the same days they
earned **0.16% a day**. Both are true, and reconciling them is the most useful thing in this
notebook. AUC scores the whole ranking, most of which is noise the strategy never touches, while
the book only ever holds the twenty names at the two ends. A model can be almost worthless as a
classifier and still useful as a sorter of extremes.

## Step 7. What is guarded against, and what is not

Three claims get made on any page like this one: no look ahead, no survivorship bias, point in
time membership. Each is checked below as a test rather than asserted.

In [ ]:
# 1. the target uses TOMORROW's return, never today's
d0, tk = px.index[1200], px.columns[0]
print("look ahead check")
print(f"  {tk} on {d0.date()}: today "
      f"{float(px.loc[d0, tk] / px.loc[px.index[1199], tk] - 1):+.5f}"
      f"   tomorrow {float(px.loc[px.index[1201], tk] / px.loc[d0, tk] - 1):+.5f}")
print(f"  stored in r1: {float(r1.loc[d0, tk]):+.5f}")

# 2. only same day index members enter the panel
snap = max(k for k in snapshots if k <= str(d0.date()))
print("membership check")
print(f"  snapshot in force on {d0.date()}: {snap}, {len(snapshots[snap])} names")
print(f"  active in the panel that day: {int(mask.loc[d0].sum())}")
print(f"  in the file but excluded that day: {int((~mask.loc[d0]).sum())}")

# 3. companies that left the index are still in the universe
ever = sorted({t for l in snapshots.values() for t in l})
today_names = set(snapshots[max(snapshots)])
gone = [t for t in ever if t not in today_names and t in px.columns]
print("survivorship check")
print(f"  {len(ever)} tickers were ever members, {len(today_names)} are members today")
print(f"  {len(gone)} left the index and stay in the panel, e.g. {gone[:8]}")

### What is not guarded against

**No transaction costs anywhere.** Krauss charged 0.05% per half turn and still kept a t-stat of
7.91. Nothing is charged here, so read every figure as gross.

**This is not current data.** Testing ends November 2015, chosen to overlap the paper. Running the
same code forward to 2025 gives roughly +0.160% a day for the forest in 2010 to 2015, +0.098% in
2015 to 2020, and +0.031% in 2020 to 2025. Reproduce that by widening `START, END` at the top and
rerunning, which takes about three times as long.

**A smaller universe.** 640 tickers with usable Yahoo history against the full index, so roughly
345 names on a given day rather than 500.

**Fewer trees, and five windows.** 60 in the forest against the thousand a paper would use, and
consecutive training sets overlap by two years, so the five window results are not fully
independent of one another.

## Step 8. PCA on the Treasury curve

Different problem, and the one where machine learning genuinely pays for itself here. PCA is
not a predictor at all, it is unsupervised compression.

Ten maturities move together almost all the time, so treating them as ten independent risks is
wasteful. Run it on daily **changes** rather than levels, because levels are non stationary and
the first component would simply track the drift of the whole rate era.

In [ ]:
tenors = ["DGS3MO","DGS6MO","DGS1","DGS2","DGS3","DGS5","DGS7","DGS10","DGS20","DGS30"]
curve  = fred(tenors).dropna()

pca = PCA(n_components=5).fit(curve.diff().dropna())
var = pca.explained_variance_ratio_ * 100
years = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]

fig, ax = plt.subplots(figsize=(11, 4.8))
for i, (c, lab) in enumerate(zip(["#2f5db0", "#d97706", "#0ca678"],
        [f"PC1 level {var[0]:.1f}%", f"PC2 slope {var[1]:.1f}%",
         f"PC3 curvature {var[2]:.1f}%"])):
    ax.plot(years, pca.components_[i], color=c, lw=2.2, marker="o", ms=5, label=lab)
ax.axhline(0, color="#dfe3ec", lw=1.2)
ax.set_xscale("log"); ax.set_xticks(years)
ax.set_xticklabels([f"{a:g}" for a in years])
ax.set_title("Treasury curve, loadings of the first three components", loc="left")
ax.set_xlabel("maturity in years"); ax.legend(frameon=False)
plt.show()

print(f"{len(curve.diff().dropna()):,} daily observations")
print("explained variance:", np.round(var, 2))
print(f"first three together: {var[:3].sum():.2f}%")

Nobody told the model to look for those shapes. PC1 comes out flat and positive across every
maturity, so it moves the whole curve. PC2 falls from the front end to the long end, tilting it.
PC3 is high at both ends and dips in the belly, bending it. Level, slope and curvature, recovered
from nothing but the covariance of daily changes.

One caveat: requiring all ten maturities drops February 2002 to February 2006, when the 30 year
bond was discontinued.

## Step 9. Lasso on a wide factor panel

Linear regression carrying a penalty on the size of its own coefficients. The L1 norm puts
corners on the constraint region, so coefficients land on exactly zero rather than merely
shrinking, and the model selects its variables while it fits them.

Target here is the forward 21 day return on SPY, with 22 sector and asset class ETFs plus six
macro series as candidates.

In [ ]:
etfs = ["XLK","XLF","XLE","XLV","XLI","XLY","XLP","XLU","XLB","XLRE","XLC",
        "IWM","EFA","EEM","TLT","IEF","HYG","LQD","GLD","USO","UUP","VNQ"]
bag = yf.download(etfs, start="2010-01-01", end=TODAY, auto_adjust=True, progress=False)
bag = bag["Close"] if "Close" in bag else bag
bag = bag.dropna(axis=1, how="all").ffill().dropna()

spy = yf.download("SPY", start="2010-01-01", end=TODAY, auto_adjust=True, progress=False)["Close"]
spy = spy.iloc[:, 0] if isinstance(spy, pd.DataFrame) else spy

mom = pd.concat({f"{c}_mom21": bag[c].pct_change(21) for c in bag.columns}, axis=1)
vol = pd.concat({f"{c}_vol21": bag[c].pct_change().rolling(21).std() for c in bag.columns}, axis=1)
macro = fred(["DGS10","DGS2","T10Y2Y","VIXCLS","BAMLH0A0HYM2","DTWEXBGS"]).reindex(bag.index).ffill()

panel = pd.concat([mom, vol, macro], axis=1).dropna()
target = (spy.shift(-21) / spy - 1).reindex(panel.index).dropna()
panel = panel.loc[target.index]

k = int(len(panel) * 0.7)
scaler = StandardScaler().fit(panel[:k])                  # fit on TRAIN only
cv = LassoCV(cv=TimeSeriesSplit(5), n_alphas=120, max_iter=20000,
             random_state=0).fit(scaler.transform(panel[:k]), target[:k])

alive = [c for c, b in zip(panel.columns, cv.coef_) if abs(b) > 1e-10]
pred  = cv.predict(scaler.transform(panel[k:]))
print(f"{len(alive)} of {panel.shape[1]} predictors survive")
print(f"survivors: {alive}")
print(f"out of sample R2 {r2_score(target[k:], pred):+.4f}   "
      f"vs {r2_score(target[k:], np.full(len(pred), target[:k].mean())):+.4f} for the mean")

In [ ]:
alphas, coefs, _ = lasso_path(scaler.transform(panel[:k]), target[:k], n_alphas=120)

fig, ax = plt.subplots(figsize=(11, 4.6))
for i in range(coefs.shape[0]):
    ax.plot(alphas, coefs[i], lw=1.3, alpha=0.9)
ax.axvline(cv.alpha_, color="#9aa1ae", ls=":", lw=1.6)
ax.axhline(0, color="#dfe3ec", lw=1.2)
ax.set_xscale("log")
ax.set_title(f"Coefficient path, {panel.shape[1]} candidate predictors", loc="left")
ax.set_xlabel("alpha")
plt.show()

## Limitations

Worth reading before trusting any number above.

**Data.** Survivorship is reduced and not eliminated: Yahoo serves usable history for roughly
three quarters of the tickers, and the missing names skew towards companies that failed, which is
exactly the group whose absence causes the bias. Membership is sampled every six months, so a
company that joined and left inside one window is invisible. Wikipedia is crowd edited and not an
index provider. Ticker reuse is unhandled. `auto_adjust=True` applies today's split and dividend
factors across the whole history. FRED series are revised, and these are current vintages.

**Method.** No transaction costs, spread, slippage or borrow anywhere, and on an edge of one point
of AUC that is the whole result rather than a rounding error. One train and test split rather than
a walk forward. Hyperparameters were chosen by hand while test results were visible, which is a
mild form of selection. One market and one era. Accuracy and AUC are not money.

These are worked examples. For work with money behind it, start from the literature:
https://davidariasfinance.com/papers/